In [128]:
import pandas as pd
import seaborn as sns
import numpy as np
import torch

In [129]:
class Feedforward(torch.nn.Module):
        def __init__(self, input_size, hidden_size, dropout):
            super(Feedforward, self).__init__()
            self.input_size = input_size
            self.hidden_size  = hidden_size
            self.fc1 = torch.nn.Linear(self.input_size, self.hidden_size)
            self.fc2 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc3 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.relu = torch.nn.ReLU()
            self.fc_out = torch.nn.Linear(self.hidden_size, 1)
            self.sigmoid = torch.nn.Sigmoid()
            self.dropout = torch.nn.Dropout(dropout)
            
        def forward(self, x):
            x = self.fc1(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc2(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc3(x)
            x = self.dropout(x)
            x = self.relu(x)
            output = self.fc_out(x)
            output = self.sigmoid(output)
            return output

In [130]:
data = pd.read_csv('/Users/nkerstingadxnet.com/Documents/Higgs/orig/atlas-higgs-challenge-2014-v2.csv')

In [131]:
data['Binary_Label'] = data['Label'].map({'s':1,'b':0})

In [76]:
data.head()

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight,Binary_Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,0.444,46.062,1.24,-2.475,113.497,0.000814,s,t,0.002653,1
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1.158,-999.000,-999.00,-999.000,46.226,0.681042,b,t,2.233584,0
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,-2.028,-999.000,-999.00,-999.000,44.251,0.715742,b,t,2.347389,0
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,-999.000,-999.000,-999.00,-999.000,-0.000,1.660654,b,t,5.446378,0
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,-999.000,-999.000,-999.00,-999.000,0.000,1.904263,b,t,6.245333,0


In [133]:
# now replace all the '-999' values with NULL so we can properly handle them
nulled_data = data.replace(-999.0, np.nan)
train_data = nulled_data.loc[nulled_data['KaggleSet'] == 't'].copy(deep=True)
public_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'b'].copy(deep=True)
private_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'v'].copy(deep=True)


In [134]:
orig_train_data = train_data.copy(deep=True)
orig_public_test_data = public_test_data.copy(deep=True)
orig_private_test_data = private_test_data.copy(deep=True)


# now replace the null values with column averages
train_data.drop('EventId', axis=1, inplace=True)
train_data.drop('Label', axis=1, inplace=True)
train_data.drop('Binary_Label', axis=1, inplace=True)
train_data.drop('Weight', axis=1, inplace=True)
train_data.drop('KaggleWeight', axis=1, inplace=True)
train_data.drop('KaggleSet', axis=1, inplace=True)
for col in train_data:
    train_data[col].fillna(train_data[col].mean(), inplace=True)
#train_data.isnull().sum()


public_test_data.drop('EventId', axis=1, inplace=True)
public_test_data.drop('Label', axis=1, inplace=True)
public_test_data.drop('Binary_Label', axis=1, inplace=True)
public_test_data.drop('Weight', axis=1, inplace=True)
public_test_data.drop('KaggleWeight', axis=1, inplace=True)
public_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in public_test_data:
    public_test_data[col].fillna(public_test_data[col].mean(), inplace=True)
#public_test_data.isnull().sum()


private_test_data.drop('EventId', axis=1, inplace=True)
private_test_data.drop('Label', axis=1, inplace=True)
private_test_data.drop('Binary_Label', axis=1, inplace=True)
private_test_data.drop('Weight', axis=1, inplace=True)
private_test_data.drop('KaggleWeight', axis=1, inplace=True)
private_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in private_test_data:
    private_test_data[col].fillna(private_test_data[col].mean(), inplace=True)
#private_test_data.isnull().sum()

In [135]:
# now let's normalize
normed_train_data = (train_data - train_data.min())/(train_data.max() - train_data.min())

normed_public_test_data = (public_test_data - public_test_data.min())/(public_test_data.max() - public_test_data.min())

normed_private_test_data = (private_test_data - private_test_data.min())/(private_test_data.max() - private_test_data.min())



In [80]:
normed_train_data.head()


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,0.109407,0.074854,0.068128,0.009869,0.107021,0.022395,0.596501,0.521549,0.014789,0.083957,...,0.455920,0.123125,0.666667,0.034326,0.738942,0.570746,0.023229,0.637778,0.106143,0.069484
1,0.128398,0.099653,0.072155,0.016983,0.282693,0.072194,0.496154,0.596238,0.000733,0.043764,...,0.195099,0.075802,0.333333,0.014878,0.580573,0.684386,0.040031,0.498684,0.499748,0.028300
2,0.095365,0.235006,0.089071,0.012570,0.282693,0.072194,0.496154,0.536888,0.003293,0.083987,...,0.152132,0.123969,0.333333,0.013067,0.728162,0.177304,0.040031,0.498684,0.499748,0.027091
3,0.114001,0.117983,0.055557,0.000146,0.282693,0.072194,0.496154,0.566472,0.000146,0.016533,...,0.509548,0.036368,0.000000,0.050269,0.499636,0.498107,0.040031,0.498684,0.499748,-0.000000
4,0.141017,0.024512,0.095662,0.005787,0.282693,0.072194,0.496154,0.672571,0.005787,0.006576,...,0.361394,0.019823,0.000000,0.050269,0.499636,0.498107,0.040031,0.498684,0.499748,0.000000


In [29]:
normed_train_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 250000 entries, 0 to 249999
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   DER_mass_MMC                 250000 non-null  float64
 1   DER_mass_transverse_met_lep  250000 non-null  float64
 2   DER_mass_vis                 250000 non-null  float64
 3   DER_pt_h                     250000 non-null  float64
 4   DER_deltaeta_jet_jet         250000 non-null  float64
 5   DER_mass_jet_jet             250000 non-null  float64
 6   DER_prodeta_jet_jet          250000 non-null  float64
 7   DER_deltar_tau_lep           250000 non-null  float64
 8   DER_pt_tot                   250000 non-null  float64
 9   DER_sum_pt                   250000 non-null  float64
 10  DER_pt_ratio_lep_tau         250000 non-null  float64
 11  DER_met_phi_centrality       250000 non-null  float64
 12  DER_lep_eta_centrality       250000 non-null  float64
 13 

In [136]:

#train_tensor_out = torch.tensor(orig_train_data['Binary_Label'].values)
#train_tensor = torch.tensor(normed_train_data.values)
train_input_data = []
for i in range(len(orig_train_data)):
   train_input_data.append([torch.tensor(normed_train_data.iloc[i], dtype=torch.float), torch.tensor(orig_train_data['Binary_Label'].iloc[i] , dtype=torch.float)])

In [137]:
valid_input_data = []
for i in range(len(orig_public_test_data)):
   valid_input_data.append([torch.tensor(normed_public_test_data.iloc[i], dtype=torch.float), torch.tensor(orig_public_test_data['Binary_Label'].iloc[i] , dtype=torch.float)])

In [138]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_input_data, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_input_data, batch_size=128, shuffle=True)

In [139]:
model = Feedforward(30, 600, 0.05)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [37]:
torch.cuda.is_available()

False

In [140]:
import copy
import time

start_time = time.time()
epoch = 20
maxvalcount = 5
minval_loss = np.inf
valcount = maxvalcount
best_model = model
for epoch in range(epoch):
    print("EPOCH: ", epoch)
    model.train()
    for i,batch in enumerate(train_dataloader):
        inputs, output = batch
        optimizer.zero_grad()
        # Forward pass
        y_pred = model(inputs)
        # Compute Loss
        loss = criterion(y_pred.squeeze(), output)
        if i % 100 == 0:
            print('Batch {}: train loss: {}'.format(i, loss.item()))
        # Backward pass
        loss.backward()
        optimizer.step()
    # compute validation loss
    model.eval()
    with torch.set_grad_enabled(False):
        val_loss = 0
        for i,batch in enumerate(valid_dataloader):
            inputs, output = batch
            y_pred = model(inputs)
            y_pred = model(inputs)
            loss = criterion(y_pred.squeeze(), output)
            val_loss += loss
            if i % 100 == 0:
                print('Validation Batch {}: loss: {}'.format(i, loss.item()))
        avg_val_loss = val_loss / len(valid_dataloader)
        print(f"AVERAGE BATCH VAL LOSS = {avg_val_loss}")
    if avg_val_loss < minval_loss:
        minval_loss = avg_val_loss
        best_model = copy.deepcopy(model)
        valcount = maxvalcount
    else:
        valcount -= 1
    if valcount == 0:
        print(f"Validation Loss failed to decrease in {maxvalcount} epochs, exiting with best model")
        break
    
end_time = time.time()
print(f"Time elapsed in training: {end_time - start_time} seconds")

EPOCH:  0
Batch 0: train loss: 0.6893978118896484
Batch 100: train loss: 0.6534700393676758
Batch 200: train loss: 0.6424453854560852
Batch 300: train loss: 0.6695780158042908
Batch 400: train loss: 0.6515956521034241
Batch 500: train loss: 0.6388377547264099
Batch 600: train loss: 0.6270337700843811
Batch 700: train loss: 0.6635515689849854
Batch 800: train loss: 0.6014732122421265
Batch 900: train loss: 0.6275156140327454
Batch 1000: train loss: 0.6535892486572266
Batch 1100: train loss: 0.5935057401657104
Batch 1200: train loss: 0.5974753499031067
Batch 1300: train loss: 0.6020992398262024
Batch 1400: train loss: 0.6448183655738831
Batch 1500: train loss: 0.6010163426399231
Batch 1600: train loss: 0.5816610455513
Batch 1700: train loss: 0.5918084979057312
Batch 1800: train loss: 0.6079549193382263
Batch 1900: train loss: 0.5744369029998779
Validation Batch 0: loss: 0.5569058060646057
Validation Batch 100: loss: 0.6172940135002136
Validation Batch 200: loss: 0.577890157699585
Validat

Batch 1300: train loss: 0.5257290601730347
Batch 1400: train loss: 0.49645230174064636
Batch 1500: train loss: 0.5808191299438477
Batch 1600: train loss: 0.528102695941925
Batch 1700: train loss: 0.49206873774528503
Batch 1800: train loss: 0.4660507142543793
Batch 1900: train loss: 0.5695800185203552
Validation Batch 0: loss: 0.597465991973877
Validation Batch 100: loss: 0.6357811689376831
Validation Batch 200: loss: 0.5783658623695374
Validation Batch 300: loss: 0.5292744040489197
Validation Batch 400: loss: 0.6580393314361572
Validation Batch 500: loss: 0.5828341245651245
Validation Batch 600: loss: 0.5687484741210938
Validation Batch 700: loss: 0.6185050010681152
AVERAGE BATCH VAL LOSS = 0.5973839163780212
EPOCH:  7
Batch 0: train loss: 0.49329739809036255
Batch 100: train loss: 0.5430585145950317
Batch 200: train loss: 0.5103341341018677
Batch 300: train loss: 0.4258919358253479
Batch 400: train loss: 0.5018779635429382
Batch 500: train loss: 0.4492100179195404
Batch 600: train los

In [141]:
torch.save(best_model, '/Users/nkerstingadxnet.com/Documents/Higgs/output/mlp.600.pt')

In [142]:
best_model.eval()
y_pred = model(torch.tensor(normed_train_data.values, dtype=torch.float)).tolist()

In [121]:
y_pred

[[0.34261271357536316],
 [0.238296240568161],
 [0.07937367260456085],
 [0.08683201670646667],
 [0.16980604827404022],
 [0.22682584822177887],
 [0.5301306843757629],
 [0.8640022873878479],
 [0.1804657280445099],
 [0.7638903260231018],
 [0.12828156352043152],
 [0.6050633788108826],
 [0.6843826174736023],
 [0.20764213800430298],
 [0.011167837306857109],
 [0.5071235299110413],
 [0.6827844381332397],
 [0.5607870221138],
 [0.06456384807825089],
 [0.03918009623885155],
 [0.03373033553361893],
 [0.28969743847846985],
 [0.2322460412979126],
 [0.7238510251045227],
 [0.27337446808815],
 [0.42022714018821716],
 [0.788186252117157],
 [0.9128872752189636],
 [0.5215741395950317],
 [0.8364686965942383],
 [0.03179359808564186],
 [0.6772437691688538],
 [0.943113386631012],
 [0.7536647319793701],
 [0.444439560174942],
 [0.34249886870384216],
 [0.8707319498062134],
 [0.45973116159439087],
 [0.095066599547863],
 [0.6292946934700012],
 [0.44185730814933777],
 [0.12578827142715454],
 [0.8793133497238159],
 [

In [143]:
import math
def AMS(s, b):
    """ Approximate Median Significance defined as:
        AMS = sqrt(
                2 { (s + b + b_r) log[1 + (s/(b+b_r))] - s}
              )        
    where b_r = 10, b = background, s = signal, log is natural logarithm """
    
    br = 10.0
    radicand = 2 *( (s+b+br) * math.log (1.0 + s/(b+br)) -s)
    if radicand < 0:
        print('radicand is negative. Exiting')
        exit()
    else:
        return math.sqrt(radicand)

In [144]:
# Training performance
y_gold = orig_train_data['Label']
y_weight = orig_train_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
print(f"TRAINING: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}")

TRAINING: (S,B) = (592.605,87199.414), AMS=2.004, Unweighted Precision = 0.577


In [145]:
y_pred = best_model(torch.tensor(normed_public_test_data.values, dtype=torch.float)).tolist()

In [146]:
y_gold = orig_public_test_data['Label']
y_weight = orig_public_test_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
print(f"VALID: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}")

VALID: (S,B) = (436.155,51380.692), AMS=1.921, Unweighted Precision = 0.606


In [147]:
y_pred = best_model(torch.tensor(normed_private_test_data.values, dtype=torch.float)).tolist()

In [148]:
y_gold = orig_private_test_data['Label']
y_weight = orig_private_test_data['KaggleWeight']
s = 0
b = 0
tp = 0
fp = 0
for i,y in enumerate(y_pred):
    if y[0] > 0.5:
        if y_gold.iloc[i] == 's':
            s += y_weight.iloc[i]
            tp += 1
        else:
            b += y_weight.iloc[i]
            fp += 1
print(f"TEST: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}")

TEST: (S,B) = (432.595,50731.178), AMS=1.918, Unweighted Precision = 0.613


In [ ]:
# stop train when val loss not decreasing three times in a row
# vary dropout
# deeper nets
# different activations

In [150]:
best_model.info()

AttributeError: 'Feedforward' object has no attribute 'info'